# DeepMind Math Synthetic CoT Dataset Builder

This notebook generates a structured dataset from the official `google-deepmind/mathematics_dataset` generator.
It generates only `train-easy`: 400,000 rows per target module, 2,000,000 rows total.
Generation is sharded and parallelized with `joblib`/`loky`; `NUM_WORKERS=None` uses all visible CPU cores.

Target modules:

- `arithmetic__add_or_sub`
- `arithmetic__add_sub_multiple`
- `comparison__pair`
- `numbers__place_value`
- `numbers__round_number`

Each accepted row is stored as:

```json
{
  "input": "...",
  "output": {
    "cot": "op=add\ncol0: ...",
    "answer": "..."
  }
}
```

In [ ]:
# Environment setup.

import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "mathematics_dataset": "mathematics_dataset",
    "sympy": "sympy",
    "pandas": "pandas",
    "tqdm": "tqdm",
    "joblib": "joblib",
    "ipykernel": "ipykernel",
}

missing = [
    pip_name
    for import_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(import_name) is None
]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are already installed.")

In [ ]:
# Parameters.

from pathlib import Path

SEED = None  # None = random base seed each run; set an int for reproducibility.
EASY_PER_MODULE = 400_000
ROWS_PER_SHARD = 5_000
NUM_WORKERS = None  # None = use all visible CPU cores.
WRITE_CSV = False  # Keep False for speed at multi-million scale.
CLEAR_EXISTING_SHARDS = True
JOBLIB_VERBOSE = 10

TARGET_MODULES = [
    "arithmetic__add_or_sub",
    "arithmetic__add_sub_multiple",
    "comparison__pair",
    "numbers__place_value",
    "numbers__round_number",
]

OUTPUT_DIR = Path("data/deepmind_math_cot")
SOURCE_NAME = "google-deepmind/mathematics_dataset"

REGIME_COUNTS = {
    "train-easy": EASY_PER_MODULE,
}
TOTAL_TARGET_ROWS = EASY_PER_MODULE * len(TARGET_MODULES)

STRICT_SYNTHETIC_VALIDATION = True
MAX_ATTEMPTS_MULTIPLIER = 80

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR.resolve())
print("Target rows:", TOTAL_TARGET_ROWS)

In [ ]:
# Imports and deterministic seeds.

import collections
import csv
import json
import math
import os
import random
import re
import secrets
import shutil
import sys
import importlib
from dataclasses import dataclass
from decimal import Decimal, ROUND_HALF_UP, InvalidOperation, localcontext
from fractions import Fraction
from pathlib import Path

import numpy as np
import pandas as pd
import sympy
from joblib import Parallel, delayed
from tqdm.auto import tqdm

# ---- compatibility patch for old mathematics_dataset + newer SymPy ----
_dioph_pkg = importlib.import_module("sympy.solvers.diophantine")
_dioph_impl = importlib.import_module("sympy.solvers.diophantine.diophantine")
_dioph_pkg.base_solution_linear = _dioph_impl.base_solution_linear

# Clear any partially imported mathematics_dataset modules from earlier failures
for _name in list(sys.modules):
    if _name == "mathematics_dataset" or _name.startswith("mathematics_dataset."):
        del sys.modules[_name]
# ----------------------------------------------------------------------

from mathematics_dataset import generate_settings
from mathematics_dataset.modules import modules

BASE_SEED = secrets.randbits(63) if SEED is None else int(SEED)
NUM_WORKERS = os.cpu_count() if NUM_WORKERS is None else int(NUM_WORKERS)
NUM_WORKERS = max(1, NUM_WORKERS)

random.seed(BASE_SEED)
np.random.seed(BASE_SEED % (2**32))
print("Base seed:", BASE_SEED)
print("CPU workers:", NUM_WORKERS)

In [ ]:
# Official generator-compatible module registry and sampling helpers.

def make_entropy_fn(level, num_levels):
    # Restrict a module entropy range to one curriculum bucket.
    lower = level / num_levels
    upper = (level + 1) / num_levels

    def modify_entropy(range_):
        assert len(range_) == 2
        length = range_[1] - range_[0]
        return (range_[0] + lower * length, range_[0] + upper * length)

    return modify_entropy


def filter_and_flatten(module_tree, target_modules):
    # Flatten the official nested module tree and keep only target names.
    target_modules = set(target_modules)
    flat = collections.OrderedDict()

    def add(submodules, prefix=None):
        for key, module_or_function in submodules.items():
            full_name = f"{prefix}__{key}" if prefix is not None else key
            if isinstance(module_or_function, dict):
                add(module_or_function, full_name)
            elif full_name in target_modules:
                flat[full_name] = module_or_function

    add(module_tree)
    return collections.OrderedDict((key, flat[key]) for key in sorted(flat))


def init_target_modules(target_modules):
    # Build only the easy training registry for large-scale generation.
    all_modules = collections.OrderedDict()
    all_modules["train-easy"] = modules.train(make_entropy_fn(0, 3))

    filtered = collections.OrderedDict()
    for regime, module_tree in all_modules.items():
        filtered[regime] = filter_and_flatten(module_tree, target_modules)
        missing = sorted(set(target_modules) - set(filtered[regime]))
        if missing:
            raise KeyError(f"{regime} is missing target modules: {missing}")
    return filtered


def sample_from_module(module_fn, max_attempts=10000):
    # Sample a problem while respecting the official length constraints.
    dropped = 0
    for _ in range(max_attempts):
        problem = module_fn()
        question = str(problem.question)
        answer = str(problem.answer)
        if len(question) > generate_settings.MAX_QUESTION_LENGTH:
            dropped += 1
            continue
        if len(answer) > generate_settings.MAX_ANSWER_LENGTH:
            dropped += 1
            continue
        return problem, dropped
    raise RuntimeError(f"Unable to sample a valid problem after {max_attempts} attempts.")


TARGET_REGISTRY = init_target_modules(TARGET_MODULES)
{regime: list(registry.keys()) for regime, registry in TARGET_REGISTRY.items()}

In [ ]:
# Shared parsing and formatting utilities for synthetic CoT builders.

NUM_RE = r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:/\d+)?"
TOKEN_RE = r"[A-Za-z_][A-Za-z_0-9]*|" + NUM_RE


class CotBuildError(ValueError):
    pass


@dataclass
class CotResult:
    cot: str
    answer: str


def clean_question(text):
    return " ".join(str(text).strip().split())


def final_sentence(text):
    parts = [part.strip() for part in re.split(r"(?<=[.?!])\s+", clean_question(text)) if part.strip()]
    return parts[-1] if parts else clean_question(text)


def extract_assignments(text):
    # Extract simple entity assignments introduced by the composition system.
    assignments = {}
    patterns = [
        rf"\bLet\s+([A-Za-z_][A-Za-z_0-9]*)\s*=\s*({NUM_RE})\s*[.]",
        rf"\bLet\s+([A-Za-z_][A-Za-z_0-9]*)\s+be\s+({NUM_RE})\s*[.]",
    ]
    for pattern in patterns:
        for name, value in re.findall(pattern, clean_question(text), flags=re.IGNORECASE):
            assignments[name] = value
    return assignments


def value_for_token(token, assignments):
    token = token.strip()
    return assignments.get(token, token)


def token_to_sympy(token, assignments=None):
    assignments = assignments or {}
    token = value_for_token(token, assignments)
    try:
        return sympy.Rational(token)
    except Exception:
        try:
            return sympy.sympify(token)
        except Exception as exc:
            raise CotBuildError(f"Cannot parse numeric token {token!r}") from exc


def token_to_decimal(token, assignments=None):
    assignments = assignments or {}
    token = value_for_token(token, assignments)
    if "/" in token:
        frac = Fraction(token)
        return Decimal(frac.numerator) / Decimal(frac.denominator)
    try:
        return Decimal(token)
    except InvalidOperation as exc:
        raise CotBuildError(f"Cannot parse decimal token {token!r}") from exc


def format_sympy(value):
    value = sympy.simplify(value)
    return str(value)


def format_decimal(value):
    if not isinstance(value, Decimal):
        value = Decimal(value)
    if value == value.to_integral_value():
        return str(value.quantize(Decimal(1)))
    return format(value.normalize(), "f")


def decimal_to_plain_text(value):
    value = Decimal(str(value))
    return format(value, "f")


def compare_answer_strings(generated, source):
    return str(generated).strip() == str(source).strip()


def normalize_expression_text(question):
    # Extract an arithmetic expression from common DeepMind templates.
    q = clean_question(question)
    q = final_sentence(q)
    q = q.rstrip("?. ")
    prefixes = [
        "What is the value of ",
        "What is ",
        "Evaluate ",
        "Calculate ",
        "Work out ",
    ]
    for prefix in prefixes:
        if q.lower().startswith(prefix.lower()):
            q = q[len(prefix):]
            break
    return q.strip()

In [ ]:
# Schema-driven pseudo-English CoT builders.

import math
from fractions import Fraction

PLACE_VALUES = collections.OrderedDict([
    ("hundred millionths", Decimal("0.00000001")),
    ("ten millionths", Decimal("0.0000001")),
    ("millionths", Decimal("0.000001")),
    ("hundred thousandths", Decimal("0.00001")),
    ("ten thousandths", Decimal("0.0001")),
    ("thousandths", Decimal("0.001")),
    ("thousandth", Decimal("0.001")),
    ("hundredths", Decimal("0.01")),
    ("hundredth", Decimal("0.01")),
    ("tenths", Decimal("0.1")),
    ("tenth", Decimal("0.1")),
    ("ones", Decimal("1")),
    ("one", Decimal("1")),
    ("units", Decimal("1")),
    ("unit", Decimal("1")),
    ("tens", Decimal("10")),
    ("ten", Decimal("10")),
    ("hundreds", Decimal("100")),
    ("hundred", Decimal("100")),
    ("thousands", Decimal("1000")),
    ("thousand", Decimal("1000")),
    ("ten thousands", Decimal("10000")),
    ("ten thousand", Decimal("10000")),
    ("hundred thousands", Decimal("100000")),
    ("hundred thousand", Decimal("100000")),
    ("millions", Decimal("1000000")),
    ("million", Decimal("1000000")),
    ("ten millions", Decimal("10000000")),
    ("ten million", Decimal("10000000")),
    ("hundred millions", Decimal("100000000")),
    ("hundred million", Decimal("100000000")),
    ("billions", Decimal("1000000000")),
    ("billion", Decimal("1000000000")),
])

INTEGER_PLACE_NAMES = {
    Decimal("1"): "ones",
    Decimal("10"): "tens",
    Decimal("100"): "hundreds",
    Decimal("1000"): "thousands",
    Decimal("10000"): "ten_thousands",
    Decimal("100000"): "hundred_thousands",
    Decimal("1000000"): "millions",
    Decimal("10000000"): "ten_millions",
    Decimal("100000000"): "hundred_millions",
    Decimal("1000000000"): "billions",
}

DECIMAL_PLACE_NAMES = {
    Decimal("0.1"): "tenths",
    Decimal("0.01"): "hundredths",
    Decimal("0.001"): "thousandths",
    Decimal("0.0001"): "ten_thousandths",
    Decimal("0.00001"): "hundred_thousandths",
    Decimal("0.000001"): "millionths",
    Decimal("0.0000001"): "ten_millionths",
    Decimal("0.00000001"): "hundred_millionths",
}


def clean_number_token(token):
    token = str(token).strip()
    if re.fullmatch(r"[+-]?\d+\.", token):
        return token[:-1]
    return token


def extract_numbers(text):
    return [clean_number_token(match) for match in re.findall(NUM_RE, clean_question(text))]


def find_place_name(text):
    lowered = clean_question(text).lower()
    for name in sorted(PLACE_VALUES, key=len, reverse=True):
        if re.search(rf"\b{re.escape(name)}\b", lowered):
            return name
    numeric_match = re.search(r"nearest\s+([+-]?\d+(?:\.\d+)?)", lowered)
    if numeric_match:
        return numeric_match.group(1)
    raise CotBuildError("No recognized place name found.")


def is_nonnegative_integer_token(token):
    return re.fullmatch(r"\+?\d+", clean_number_token(token)) is not None


def is_integer_token(token):
    return re.fullmatch(r"[+-]?\d+", clean_number_token(token)) is not None


def compact_binary(left, op, right):
    return f"{clean_number_token(left)}{op}{clean_number_token(right)}"


def column_add_lines(left_token, right_token):
    left_digits = list(map(int, str(int(clean_number_token(left_token)))[::-1]))
    right_digits = list(map(int, str(int(clean_number_token(right_token)))[::-1]))
    width = max(len(left_digits), len(right_digits))
    carry = 0
    lines = ["op=add"]
    for col in range(width):
        a = left_digits[col] if col < len(left_digits) else 0
        b = right_digits[col] if col < len(right_digits) else 0
        carry_in = carry
        total = a + b + carry_in
        parts = [str(a), str(b)]
        if carry_in:
            parts.append(str(carry_in))
        expr = "+".join(parts)
        carry = total // 10
        write = total % 10
        if col == width - 1 and carry:
            lines.append(f"col{col}: {expr}={total} | write={write} carry={carry}")
        elif col == width - 1:
            lines.append(f"col{col}: {expr}={total}")
        else:
            lines.append(f"col{col}: {expr}={total} | write={write} carry={carry}")
    if carry:
        lines.append(f"col{width}: carry={carry}")
    return lines


def column_sub_lines(left_token, right_token):
    left_digits = list(map(int, str(int(clean_number_token(left_token)))[::-1]))
    right_digits = list(map(int, str(int(clean_number_token(right_token)))[::-1]))
    width = max(len(left_digits), len(right_digits))
    borrow = 0
    lines = ["op=sub"]
    for col in range(width):
        a = left_digits[col] if col < len(left_digits) else 0
        b = right_digits[col] if col < len(right_digits) else 0
        effective = a - borrow
        if effective < b:
            value = effective + 10 - b
            prefix = f"{a}-{borrow}-{b}" if borrow else f"{a}-{b}"
            lines.append(f"col{col}: {prefix} needs_borrow | {effective + 10}-{b}={value} borrow=1")
            borrow = 1
        else:
            value = effective - b
            expr = f"{a}-{borrow}-{b}" if borrow else f"{a}-{b}"
            lines.append(f"col{col}: {expr}={value}")
            borrow = 0
    return lines


def align_decimal_tokens(left_token, right_token):
    def split_decimal(token):
        token = clean_number_token(token)
        sign = ""
        if token.startswith(("-", "+")):
            sign, token = token[0], token[1:]
        whole, dot, frac = token.partition(".")
        return sign, whole or "0", frac if dot else ""

    l_sign, l_whole, l_frac = split_decimal(left_token)
    r_sign, r_whole, r_frac = split_decimal(right_token)
    width = max(len(l_frac), len(r_frac))
    left = f"{l_sign}{l_whole}.{l_frac.ljust(width, '0')}" if width else f"{l_sign}{l_whole}"
    right = f"{r_sign}{r_whole}.{r_frac.ljust(width, '0')}" if width else f"{r_sign}{r_whole}"
    return left, right


def binary_add_sub_schema(left_token, op, right_token):
    left_token = clean_number_token(left_token)
    right_token = clean_number_token(right_token)
    left = token_to_decimal(left_token)
    right = token_to_decimal(right_token)
    result = left + right if op == "+" else left - right
    answer = format_decimal(result)

    if op == "+" and is_nonnegative_integer_token(left_token) and is_nonnegative_integer_token(right_token):
        return "\n".join(column_add_lines(left_token, right_token)), answer
    if op == "-" and is_nonnegative_integer_token(left_token) and is_nonnegative_integer_token(right_token):
        if int(left_token) >= int(right_token):
            return "\n".join(column_sub_lines(left_token, right_token)), answer
    if op == "-" and is_nonnegative_integer_token(left_token) and re.fullmatch(r"-\d+", right_token):
        normalized_right = right_token[1:]
        lines = [f"normalize: {compact_binary(left_token, '-', right_token)} -> {compact_binary(left_token, '+', normalized_right)}"]
        lines.extend(column_add_lines(left_token, normalized_right))
        return "\n".join(lines), answer
    if op == "+" and is_nonnegative_integer_token(left_token) and re.fullmatch(r"-\d+", right_token):
        normalized_right = right_token[1:]
        if int(left_token) >= int(normalized_right):
            lines = [f"normalize: {compact_binary(left_token, '+', right_token)} -> {compact_binary(left_token, '-', normalized_right)}"]
            lines.extend(column_sub_lines(left_token, normalized_right))
            return "\n".join(lines), answer
    if "." in left_token or "." in right_token:
        left_aligned, right_aligned = align_decimal_tokens(left_token, right_token)
        op_name = "add" if op == "+" else "sub"
        return "\n".join([
            f"op={op_name}",
            f"align: {left_token} {op} {right_token} -> {left_aligned} {op} {right_aligned}",
            f"compute: {left_aligned}{op}{right_aligned}={answer}",
        ]), answer
    op_name = "add" if op == "+" else "sub"
    return "\n".join([f"op={op_name}", f"compute: {compact_binary(left_token, op, right_token)}={answer}"]), answer


def build_add_or_sub_cot(question, source_answer):
    prompt = final_sentence(question)
    for name, value in extract_assignments(question).items():
        prompt = re.sub(rf"\b{re.escape(name)}\b", value, prompt)
    patterns = [
        (rf"({NUM_RE})\s*([+-])\s*({NUM_RE})", "symbolic"),
        (rf"({NUM_RE})\s+plus\s+({NUM_RE})", "plus"),
        (rf"Add\s+({NUM_RE})\s+and\s+({NUM_RE})", "plus"),
        (rf"Add together\s+({NUM_RE})\s+and\s+({NUM_RE})", "plus"),
        (rf"Put together\s+({NUM_RE})\s+and\s+({NUM_RE})", "plus"),
        (rf"Sum\s+({NUM_RE})\s+and\s+({NUM_RE})", "plus"),
        (rf"Total of\s+({NUM_RE})\s+and\s+({NUM_RE})", "plus"),
        (rf"({NUM_RE})\s+minus\s+({NUM_RE})", "minus"),
        (rf"({NUM_RE})\s+take away\s+({NUM_RE})", "minus"),
        (rf"Subtract\s+({NUM_RE})\s+from\s+({NUM_RE})", "subtract_from"),
        (rf"({NUM_RE})\s+less than\s+({NUM_RE})", "less_than"),
        (rf"(?:distance|difference)\s+between\s+({NUM_RE})\s+and\s+({NUM_RE})", "difference"),
    ]
    for pattern, kind in patterns:
        match = re.search(pattern, prompt, flags=re.IGNORECASE)
        if not match:
            continue
        if kind == "symbolic":
            left, op, right = [clean_number_token(x) for x in match.groups()]
            cot, answer = binary_add_sub_schema(left, op, right)
        elif kind == "plus":
            left, right = [clean_number_token(x) for x in match.groups()]
            cot, answer = binary_add_sub_schema(left, "+", right)
        elif kind == "minus":
            left, right = [clean_number_token(x) for x in match.groups()]
            cot, answer = binary_add_sub_schema(left, "-", right)
        elif kind in {"subtract_from", "less_than"}:
            subtrahend, minuend = [clean_number_token(x) for x in match.groups()]
            cot, answer = binary_add_sub_schema(minuend, "-", subtrahend)
        else:
            left, right = [clean_number_token(x) for x in match.groups()]
            answer = format_decimal(abs(token_to_decimal(left) - token_to_decimal(right)))
            cot = "\n".join(["op=abs_diff", f"compute: |{left}-{right}|={answer}"])
        return CotResult(cot=cot, answer=answer)
    raise CotBuildError(f"Unsupported add_or_sub template: {question}")


def compact_expr(expr):
    return re.sub(r"\s+", "", expr)


def normalize_double_minus(expr):
    return expr.replace("--", "+")


def signed_int_tokens(expr):
    if not re.fullmatch(r"[0-9+\-]+", expr):
        raise CotBuildError(f"Unsupported flat add/sub expression: {expr!r}")
    return re.findall(r"[+-]?\d+", expr)


def format_pair_expr(a, b):
    b = str(b)
    if b.startswith("+"):
        return f"{a}+{b[1:]}"
    if b.startswith("-"):
        return f"{a}{b}"
    return f"{a}+{b}"


def reduce_flat_expression(expr):
    lines = []
    normalized = normalize_double_minus(expr)
    if normalized != expr:
        lines.append(f"normalize: {expr} -> {normalized}")
    tokens = signed_int_tokens(normalized)
    if not tokens:
        raise CotBuildError(f"Could not reduce expression: {expr!r}")
    while len(tokens) > 1:
        a = int(tokens[0])
        b = int(tokens[1])
        value = a + b
        lines.append(f"reduce: {format_pair_expr(tokens[0], tokens[1])}={value}")
        tokens = [str(value)] + tokens[2:]
    return lines, tokens[0]


def build_add_sub_multiple_cot(question, source_answer):
    expression = normalize_expression_text(question)
    for name, value in extract_assignments(question).items():
        expression = re.sub(rf"\b{re.escape(name)}\b", f"({value})", expression)
    expression = compact_expr(expression)
    if not re.fullmatch(r"[0-9+\-()]+", expression):
        raise CotBuildError(f"Unsupported add_sub_multiple expression: {expression!r}")
    lines = [f"expr={expression}"]
    working = expression
    while True:
        match = re.search(r"\([^()]*\)", working)
        if not match:
            break
        inner = match.group(0)[1:-1]
        inner_lines, value = reduce_flat_expression(inner)
        lines.extend(inner_lines)
        working = working[:match.start()] + value + working[match.end():]
    final_lines, answer = reduce_flat_expression(working)
    lines.extend(final_lines)
    return CotResult(cot="\n".join(lines), answer=answer)


def parse_pair_tokens_schema(prompt):
    token = rf"({TOKEN_RE})"
    patterns = [
        (rf"^{token}\s*(<=|>=|<|>|=|!=)\s*{token}\?$", "symbol_bool"),
        (rf"Which is (?:bigger|greater):\s*{token}\s+or\s+{token}\?", "which_greater"),
        (rf"Which is smaller:\s*{token}\s+or\s+{token}\?", "which_smaller"),
        (rf"Is\s+{token}\s*(<=|>=|<|>|=|!=)\s*{token}\?", "symbol_bool"),
        (rf"Does\s+{token}\s*=\s*{token}\?", "equal_bool"),
        (rf"Are\s+{token}\s+and\s+{token}\s+equal\?", "equal_bool"),
        (rf"Is\s+{token}\s+equal to\s+{token}\?", "equal_bool"),
        (rf"Do\s+{token}\s+and\s+{token}\s+have the same value\?", "equal_bool"),
        (rf"Is\s+{token}\s+not equal to\s+{token}\?", "not_equal_bool"),
        (rf"Are\s+{token}\s+and\s+{token}\s+(?:unequal|nonequal|non-equal)\?", "not_equal_bool"),
        (rf"Do\s+{token}\s+and\s+{token}\s+have different values\?", "not_equal_bool"),
        (rf"Is\s+{token}\s+less than or equal to\s+{token}\?", "le_bool"),
        (rf"Is\s+{token}\s+at most(?: as big as)?\s+{token}\?", "le_bool"),
        (rf"Is\s+{token}\s+(?:less than|smaller than)\s+{token}\?", "lt_bool"),
        (rf"Is\s+{token}\s+greater than or equal to\s+{token}\?", "ge_bool"),
        (rf"Is\s+{token}\s+at least(?: as big as)?\s+{token}\?", "ge_bool"),
        (rf"Is\s+{token}\s+(?:greater than|bigger than)\s+{token}\?", "gt_bool"),
    ]
    for pattern, kind in patterns:
        match = re.search(pattern, prompt, flags=re.IGNORECASE)
        if match:
            return kind, match.groups()
    raise CotBuildError(f"Unsupported comparison pair template: {prompt}")


def relation_symbol(left, right):
    if sympy.Gt(left, right):
        return ">"
    if sympy.Lt(left, right):
        return "<"
    return "="


def comparison_detail_lines(left_token, right_token, assignments=None):
    left_display = clean_number_token(value_for_token(left_token, assignments))
    right_display = clean_number_token(value_for_token(right_token, assignments))
    left_value = token_to_sympy(left_token, assignments)
    right_value = token_to_sympy(right_token, assignments)
    rel = relation_symbol(left_value, right_value)
    lines = [f"compare: {left_display} vs {right_display}"]

    if "/" in left_display and "/" in right_display:
        left_frac = Fraction(left_display)
        right_frac = Fraction(right_display)
        common = math.lcm(left_frac.denominator, right_frac.denominator)
        left_num = left_frac.numerator * (common // left_frac.denominator)
        right_num = right_frac.numerator * (common // right_frac.denominator)
        lines.extend([f"common_denominator={common}", f"left={left_num}/{common}", f"right={right_num}/{common}", f"result: {left_display}{rel}{right_display}"])
        return lines, rel

    if is_integer_token(left_display) and is_integer_token(right_display):
        left_int = int(left_display)
        right_int = int(right_display)
        left_sign = "-" if left_int < 0 else "+"
        right_sign = "-" if right_int < 0 else "+"
        if left_sign != right_sign:
            lines.append(f"signs: {left_display}={left_sign} {right_display}={right_sign}")
            lines.append(f"result: {left_display}{rel}{right_display}")
            return lines, rel
        left_digits = str(abs(left_int))
        right_digits = str(abs(right_int))
        lines.append(f"digits: {left_display}={len(left_digits)} {right_display}={len(right_digits)}")
        if len(left_digits) != len(right_digits):
            lines.append(f"result: {left_display}{rel}{right_display}")
            return lines, rel
        for idx, (left_digit, right_digit) in enumerate(zip(left_digits, right_digits)):
            if left_digit == right_digit:
                lines.append(f"d{idx}: {left_digit}={right_digit}")
            else:
                digit_rel = ">" if left_digit > right_digit else "<"
                lines.append(f"d{idx}: {left_digit}{digit_rel}{right_digit}")
                break
        lines.append(f"result: {left_display}{rel}{right_display}")
        return lines, rel

    left_dec = Decimal(str(token_to_decimal(left_token, assignments)))
    right_dec = Decimal(str(token_to_decimal(right_token, assignments)))
    left_text = format(left_dec, "f")
    right_text = format(right_dec, "f")
    left_whole, _, left_frac = left_text.partition(".")
    right_whole, _, right_frac = right_text.partition(".")
    lines.append(f"integer: {left_whole} vs {right_whole}")
    if Decimal(left_whole) == Decimal(right_whole):
        width = max(len(left_frac), len(right_frac))
        for idx, (left_digit, right_digit) in enumerate(zip(left_frac.ljust(width, "0"), right_frac.ljust(width, "0"))):
            if left_digit == right_digit:
                lines.append(f"frac{idx}: {left_digit}={right_digit}")
            else:
                digit_rel = ">" if left_digit > right_digit else "<"
                lines.append(f"frac{idx}: {left_digit}{digit_rel}{right_digit}")
                break
    lines.append(f"result: {left_display}{rel}{right_display}")
    return lines, rel


def build_comparison_pair_cot(question, source_answer):
    assignments = extract_assignments(question)
    prompt = final_sentence(question).replace("\u2264", "<=").replace("\u2265", ">=").replace("\u2260", "!=")
    kind, groups = parse_pair_tokens_schema(prompt)
    if kind == "symbol_bool":
        left_token, op, right_token = groups
    else:
        left_token, right_token = groups
        op = {"equal_bool": "=", "not_equal_bool": "!=", "le_bool": "<=", "lt_bool": "<", "ge_bool": ">=", "gt_bool": ">"}.get(kind)
    left_token = clean_number_token(left_token)
    right_token = clean_number_token(right_token)
    left_value = token_to_sympy(left_token, assignments)
    right_value = token_to_sympy(right_token, assignments)
    lines, _ = comparison_detail_lines(left_token, right_token, assignments)
    if kind == "which_greater":
        answer = left_token if sympy.Gt(left_value, right_value) else right_token
    elif kind == "which_smaller":
        answer = left_token if sympy.Lt(left_value, right_value) else right_token
    else:
        operations = {"<": sympy.Lt, "<=": sympy.Le, ">": sympy.Gt, ">=": sympy.Ge, "=": sympy.Eq, "!=": sympy.Ne}
        query = f"{value_for_token(left_token, assignments)}{op}{value_for_token(right_token, assignments)}"
        answer = str(bool(operations[op](left_value, right_value)))
        lines.insert(1, f"query: {query}")
    return CotResult(cot="\n".join(lines), answer=answer)


def place_name_for_value(place):
    if place >= 1:
        return INTEGER_PLACE_NAMES.get(place, f"{format_decimal(place)}s")
    return DECIMAL_PLACE_NAMES.get(place, format_decimal(place))


def schema_place_label(place):
    if place >= 1:
        return f"{int(place)}s"
    return place_name_for_value(place)


def decimal_to_plain_text(value):
    value = Decimal(str(value))
    return format(value, "f")


def digit_at_place(number_token, place):
    number = token_to_decimal(number_token)
    signless = abs(number)
    text = decimal_to_plain_text(signless)
    integer_part, _, fractional_part = text.partition(".")
    integer_part = integer_part or "0"
    if place >= 1:
        index_from_right = int(math.log10(int(place))) if place != 1 else 0
        if index_from_right >= len(integer_part):
            return 0
        return int(integer_part[-1 - index_from_right])
    index = abs(int(place.log10())) - 1
    if index >= len(fractional_part):
        return 0
    return int(fractional_part[index])


def scan_place_line(number_token, place):
    number = abs(token_to_decimal(number_token))
    text = decimal_to_plain_text(number)
    integer_part, _, fractional_part = text.partition(".")
    if place >= 1:
        max_power = int(math.log10(int(place))) if place != 1 else 0
        parts = []
        for power in range(max_power + 1):
            current_place = Decimal(10) ** power
            label = place_name_for_value(current_place)
            idx = len(integer_part) - 1 - power
            digit = integer_part[idx] if idx >= 0 else "0"
            parts.append(f"{label}={digit}")
        return "scan_right: " + " ".join(parts)
    max_idx = abs(int(place.log10())) - 1
    parts = []
    for idx in range(max_idx + 1):
        current_place = Decimal(1).scaleb(-(idx + 1))
        label = place_name_for_value(current_place)
        digit = fractional_part[idx] if idx < len(fractional_part) else "0"
        parts.append(f"{label}={digit}")
    return "scan_decimal: " + " ".join(parts)


def build_place_value_cot(question, source_answer):
    prompt = final_sentence(question)
    place_name = find_place_name(prompt)
    place = PLACE_VALUES[place_name] if place_name in PLACE_VALUES else Decimal(place_name)
    numbers_found = extract_numbers(prompt)
    if not numbers_found:
        raise CotBuildError(f"No number found in place_value question: {question}")
    number_token = numbers_found[-1]
    digit = digit_at_place(number_token, place)
    asks_value = bool(re.search(r"\bvalue\b|\bworth\b", prompt, flags=re.IGNORECASE))
    value = Decimal(digit) * place
    answer = format_decimal(value) if asks_value else str(digit)
    lines = [f"number={number_token}", f"place={place_name.replace(' ', '_')}", scan_place_line(number_token, place)]
    if asks_value:
        lines.append(f"value: {digit}*{format_decimal(place)}={answer}")
    else:
        lines.append(f"digit={digit}")
    return CotResult(cot="\n".join(lines), answer=answer)


def quantize_to_place(number, place):
    with localcontext() as ctx:
        ctx.prec = max(50, len(str(number)) + 20)
        if place >= 1:
            scaled = (number / place).quantize(Decimal("1"), rounding=ROUND_HALF_UP)
            return scaled * place
        return number.quantize(place, rounding=ROUND_HALF_UP)


def build_round_number_cot(question, source_answer):
    prompt = final_sentence(question)
    numbers_found = extract_numbers(prompt)
    if not numbers_found:
        raise CotBuildError(f"No number found in round_number question: {question}")
    number_token = numbers_found[0]
    number = token_to_decimal(number_token)
    lowered = prompt.lower()
    decimal_places_match = re.search(r"(\d+)\s+decimal places?", lowered)
    if decimal_places_match:
        decimals = int(decimal_places_match.group(1))
        place = Decimal(1).scaleb(-decimals)
    else:
        place_name = find_place_name(prompt)
        place = PLACE_VALUES[place_name] if place_name in PLACE_VALUES else Decimal(place_name)
    inspect_place = place / Decimal(10)
    inspect_digit = digit_at_place(number_token, inspect_place)
    direction = "up" if inspect_digit >= 5 else "down"
    rounded = quantize_to_place(number, place)
    answer = format_decimal(rounded)
    lines = [
        f"number={number_token}",
        f"target={schema_place_label(place)}",
        f"inspect={schema_place_label(inspect_place)}",
        f"digit={inspect_digit}",
        f"round={direction}",
        f"result={answer}",
    ]
    return CotResult(cot="\n".join(lines), answer=answer)


COT_BUILDERS = {
    "arithmetic__add_or_sub": build_add_or_sub_cot,
    "arithmetic__add_sub_multiple": build_add_sub_multiple_cot,
    "comparison__pair": build_comparison_pair_cot,
    "numbers__place_value": build_place_value_cot,
    "numbers__round_number": build_round_number_cot,
}


def build_cot(module_name, question, source_answer):
    result = COT_BUILDERS[module_name](question, source_answer)
    if STRICT_SYNTHETIC_VALIDATION and not compare_answer_strings(result.answer, source_answer):
        raise CotBuildError(
            f"Synthetic answer mismatch for {module_name}: generated={result.answer!r}, source={source_answer!r}, question={question!r}"
        )
    return result


In [ ]:
# Manual parser/CoT smoke tests with representative schema fragments.

manual_examples = [
    ("arithmetic__add_or_sub", "Calculate 483 + 129.", "612", ["op=add", "col0: 3+9=12", "write=2", "carry=1"]),
    ("arithmetic__add_or_sub", "Subtract -4 from 10.", "14", ["normalize: 10--4 -> 10+4", "op=add"]),
    ("arithmetic__add_sub_multiple", "What is -2 + -2 - (1 - (-9 - -13))?", "-1", ["expr=-2+-2-(1-(-9--13))", "normalize: -9--13 -> -9+13", "reduce: -4+3=-1"]),
    ("comparison__pair", "362 > 62?", "True", ["digits: 362=3 62=2", "result: 362>62"]),
    ("comparison__pair", "362 > 361?", "True", ["d0: 3=3", "d1: 6=6", "d2: 2>1"]),
    ("comparison__pair", "Which is bigger: 3/4 or 2/3?", "3/4", ["common_denominator=12", "left=9/12", "right=8/12"]),
    ("numbers__place_value", "What is the hundreds digit of 26266?", "2", ["scan_right: ones=6 tens=6 hundreds=2", "digit=2"]),
    ("numbers__place_value", "What is the value of the hundreds digit of 1234?", "200", ["value: 2*100=200"]),
    ("numbers__round_number", "Round 3478 to the nearest hundred.", "3500", ["target=100s", "inspect=10s", "digit=7", "round=up"]),
    ("numbers__round_number", "What is 12.345 rounded to 2 decimal places?", "12.35", ["target=hundredths", "inspect=thousandths", "digit=5", "round=up"]),
]

for module_name, question, expected, fragments in manual_examples:
    result = build_cot(module_name, question, expected)
    assert result.answer == expected, (module_name, question, result.answer, expected)
    for fragment in fragments:
        assert fragment in result.cot, (module_name, question, fragment, result.cot)
    assert "Therefore" not in result.cot

print(f"Manual schema smoke tests passed: {len(manual_examples)} examples")

In [ ]:
# Compatibility shim for old mathematics_dataset internals on Python 3.12+.
# Python 3.12 stopped auto-converting integral floats in random.randrange(),
# so old code that passes values like 5.0 now raises TypeError.
import numbers
import random

if not hasattr(random.Random, "_math_dataset_orig_randrange"):
    random.Random._math_dataset_orig_randrange = random.Random.randrange
if not hasattr(random.Random, "_math_dataset_orig_randint"):
    random.Random._math_dataset_orig_randint = random.Random.randint
_ORIG_RANDOM_CLS_RANDRANGE = random.Random._math_dataset_orig_randrange
_ORIG_RANDOM_CLS_RANDINT = random.Random._math_dataset_orig_randint


def _coerce_integral_random_arg(x):
    if isinstance(x, numbers.Integral):
        return int(x)
    if isinstance(x, numbers.Real):
        xf = float(x)
        if xf.is_integer():
            return int(xf)
        raise TypeError(f"Non-integral value passed to random integer API: {x!r}")
    return x


def _patched_random_cls_randrange(self, start, stop=None, step=1):
    start = _coerce_integral_random_arg(start)
    if stop is not None:
        stop = _coerce_integral_random_arg(stop)
    step = _coerce_integral_random_arg(step)
    return _ORIG_RANDOM_CLS_RANDRANGE(self, start, stop, step)


def _patched_random_cls_randint(self, a, b):
    a = _coerce_integral_random_arg(a)
    b = _coerce_integral_random_arg(b)
    return _ORIG_RANDOM_CLS_RANDINT(self, a, b)


# Patch both Random methods and module-level aliases in the notebook process.
random.Random.randrange = _patched_random_cls_randrange
random.Random.randint = _patched_random_cls_randint
random.randrange = lambda start, stop=None, step=1: random._inst.randrange(start, stop, step)
random.randint = lambda a, b: random._inst.randint(a, b)

# Parallel sharded generation. Each worker writes one independent shard, then
# the main process merges shards into the final JSONL. This avoids one shared
# writer bottleneck and lets loky use notebook-defined functions on Windows.
jsonl_path = OUTPUT_DIR / "deepmind_math_cot.jsonl"
csv_path = OUTPUT_DIR / "deepmind_math_cot_flat.csv"
summary_path = OUTPUT_DIR / "dataset_summary.json"
failure_path = OUTPUT_DIR / "cot_failure_examples.json"
shards_dir = OUTPUT_DIR / "shards"
shard_manifest_path = OUTPUT_DIR / "shard_manifest.jsonl"
csv_fieldnames = ["input", "cot", "answer"]
required_top_level = {"input", "output"}


def validate_row(row):
    assert set(row.keys()) == required_top_level
    assert row["input"]
    assert row["output"]["cot"]
    assert row["output"]["answer"]
    assert "Therefore" not in row["output"]["cot"]


def flatten_row(row):
    return {
        "input": row["input"],
        "cot": row["output"]["cot"],
        "answer": row["output"]["answer"],
    }


def _patch_worker_runtime(seed):
    import importlib
    import numbers
    import random
    import sys

    import numpy as np
    import sympy

    # SymPy compatibility for old mathematics_dataset releases.
    try:
        dioph_pkg = importlib.import_module("sympy.solvers.diophantine")
        dioph_impl = importlib.import_module("sympy.solvers.diophantine.diophantine")
        dioph_pkg.base_solution_linear = dioph_impl.base_solution_linear
    except Exception:
        pass

    if not hasattr(random.Random, "_math_dataset_orig_randrange"):
        random.Random._math_dataset_orig_randrange = random.Random.randrange
    if not hasattr(random.Random, "_math_dataset_orig_randint"):
        random.Random._math_dataset_orig_randint = random.Random.randint
    orig_randrange = random.Random._math_dataset_orig_randrange
    orig_randint = random.Random._math_dataset_orig_randint

    def coerce_integral_random_arg(x):
        if isinstance(x, numbers.Integral):
            return int(x)
        if isinstance(x, numbers.Real):
            xf = float(x)
            if xf.is_integer():
                return int(xf)
            raise TypeError(f"Non-integral value passed to random integer API: {x!r}")
        return x

    def patched_randrange(self, start, stop=None, step=1):
        start = coerce_integral_random_arg(start)
        if stop is not None:
            stop = coerce_integral_random_arg(stop)
        step = coerce_integral_random_arg(step)
        return orig_randrange(self, start, stop, step)

    def patched_randint(self, a, b):
        a = coerce_integral_random_arg(a)
        b = coerce_integral_random_arg(b)
        return orig_randint(self, a, b)

    random.Random.randrange = patched_randrange
    random.Random.randint = patched_randint
    random.randrange = lambda start, stop=None, step=1: random._inst.randrange(start, stop, step)
    random.randint = lambda a, b: random._inst.randint(a, b)
    random.seed(seed)
    np.random.seed(seed % (2**32))

    # Ensure imports see the patched runtime.
    for name in list(sys.modules):
        if name == "mathematics_dataset" or name.startswith("mathematics_dataset."):
            del sys.modules[name]

    from mathematics_dataset import generate_settings as worker_generate_settings
    from mathematics_dataset.modules import modules as worker_modules

    return worker_generate_settings, worker_modules


def _worker_module_fn(worker_modules, regime, module_name):
    if regime != "train-easy":
        raise ValueError(f"Unsupported regime for this notebook: {regime}")
    module_tree = worker_modules.train(make_entropy_fn(0, 3))
    registry = filter_and_flatten(module_tree, [module_name])
    return registry[module_name]


def _sample_from_module_worker(module_fn, worker_generate_settings, max_attempts=1000):
    dropped = 0
    generator_failures = 0
    generator_failure_examples = []
    for _ in range(max_attempts):
        try:
            problem = module_fn()
        except Exception as exc:
            generator_failures += 1
            if len(generator_failure_examples) < 5:
                generator_failure_examples.append({"type": type(exc).__name__, "error": str(exc)})
            continue
        question = str(problem.question)
        answer = str(problem.answer)
        if len(question) > worker_generate_settings.MAX_QUESTION_LENGTH:
            dropped += 1
            continue
        if len(answer) > worker_generate_settings.MAX_ANSWER_LENGTH:
            dropped += 1
            continue
        return problem, dropped, generator_failures, generator_failure_examples
    raise RuntimeError(f"Unable to sample a valid problem after {max_attempts} attempts.")


def generate_shard(job):
    worker_generate_settings, worker_modules = _patch_worker_runtime(job["seed"])
    module_fn = _worker_module_fn(worker_modules, job["regime"], job["module"])

    jsonl_shard_path = Path(job["jsonl_path"])
    csv_shard_path = Path(job["csv_path"]) if job["write_csv"] else None
    jsonl_shard_path.parent.mkdir(parents=True, exist_ok=True)
    if csv_shard_path is not None:
        csv_shard_path.parent.mkdir(parents=True, exist_ok=True)

    accepted_here = 0
    attempts = 0
    duplicates_skipped = 0
    length_dropped = 0
    generator_failures = 0
    generator_failure_examples = []
    cot_failures = 0
    cot_failure_examples = []
    review_sample = None
    seen = set()
    max_attempts = max(1000, job["rows"] * job["max_attempts_multiplier"])

    csv_file = None
    csv_writer = None
    try:
        if csv_shard_path is not None:
            csv_file = csv_shard_path.open("w", encoding="utf-8", newline="")
            csv_writer = csv.DictWriter(csv_file, fieldnames=csv_fieldnames)
            csv_writer.writeheader()

        with jsonl_shard_path.open("w", encoding="utf-8") as jsonl_file:
            while accepted_here < job["rows"] and attempts < max_attempts:
                attempts += 1
                problem, dropped, sample_generator_failures, sample_generator_failure_examples = _sample_from_module_worker(module_fn, worker_generate_settings)
                length_dropped += dropped
                generator_failures += sample_generator_failures
                for failure in sample_generator_failure_examples:
                    if len(generator_failure_examples) < 5:
                        generator_failure_examples.append(failure)

                question = clean_question(problem.question)
                source_answer = str(problem.answer)
                if question in seen:
                    duplicates_skipped += 1
                    continue

                try:
                    cot_result = build_cot(job["module"], question, source_answer)
                except CotBuildError as exc:
                    cot_failures += 1
                    if len(cot_failure_examples) < 5:
                        cot_failure_examples.append(
                            {
                                "module": job["module"],
                                "regime": job["regime"],
                                "question": question,
                                "source_answer": source_answer,
                                "error": str(exc),
                            }
                        )
                    continue

                row = {
                    "input": question,
                    "output": {
                        "cot": cot_result.cot,
                        "answer": cot_result.answer,
                    },
                }
                validate_row(row)
                seen.add(question)
                jsonl_file.write(json.dumps(row, ensure_ascii=False) + "\n")
                if csv_writer is not None:
                    csv_writer.writerow(flatten_row(row))

                if review_sample is None:
                    review_sample = {
                        "module": job["module"],
                        "regime": job["regime"],
                        "input": row["input"],
                        "cot": row["output"]["cot"],
                        "answer": row["output"]["answer"],
                    }
                accepted_here += 1
    finally:
        if csv_file is not None:
            csv_file.close()

    if accepted_here < job["rows"]:
        raise RuntimeError(
            f"Only accepted {accepted_here}/{job['rows']} rows for "
            f"{job['regime']}/{job['module']} shard {job['shard_index']}."
        )

    return {
        "module": job["module"],
        "regime": job["regime"],
        "shard_index": job["shard_index"],
        "global_shard_index": job["global_shard_index"],
        "seed": job["seed"],
        "rows_requested": job["rows"],
        "rows_written": accepted_here,
        "attempts": attempts,
        "duplicates_skipped": duplicates_skipped,
        "length_dropped": length_dropped,
        "generator_failures": generator_failures,
        "generator_failure_examples": generator_failure_examples,
        "cot_failures": cot_failures,
        "cot_failure_examples": cot_failure_examples,
        "review_sample": review_sample,
        "jsonl_path": str(jsonl_shard_path),
        "csv_path": str(csv_shard_path) if csv_shard_path is not None else None,
    }


def build_shard_jobs():
    seed_rng = random.Random(BASE_SEED)
    jobs = []
    global_shard_index = 0
    for regime, target_count in REGIME_COUNTS.items():
        for module_name in TARGET_MODULES:
            remaining = target_count
            shard_index = 0
            while remaining > 0:
                rows = min(ROWS_PER_SHARD, remaining)
                shard_seed = seed_rng.randrange(0, 2**63)
                jsonl_shard_path = shards_dir / f"module={module_name}" / f"shard-{shard_index:06d}.jsonl"
                csv_shard_path = shards_dir / f"module={module_name}" / f"shard-{shard_index:06d}.csv"
                jobs.append(
                    {
                        "module": module_name,
                        "regime": regime,
                        "rows": rows,
                        "seed": shard_seed,
                        "shard_index": shard_index,
                        "global_shard_index": global_shard_index,
                        "jsonl_path": str(jsonl_shard_path),
                        "csv_path": str(csv_shard_path),
                        "write_csv": WRITE_CSV,
                        "max_attempts_multiplier": MAX_ATTEMPTS_MULTIPLIER,
                    }
                )
                remaining -= rows
                shard_index += 1
                global_shard_index += 1
    return jobs


if CLEAR_EXISTING_SHARDS and shards_dir.exists():
    shutil.rmtree(shards_dir)
for path in [jsonl_path, csv_path, summary_path, failure_path, shard_manifest_path]:
    if path.exists():
        path.unlink()

shard_jobs = build_shard_jobs()
print(f"Shard jobs: {len(shard_jobs)}")
print(f"Rows per shard: {ROWS_PER_SHARD}")
print(f"Requested workers: {NUM_WORKERS}")

shard_results = Parallel(
    n_jobs=NUM_WORKERS,
    backend="loky",
    verbose=JOBLIB_VERBOSE,
    batch_size=1,
)(delayed(generate_shard)(job) for job in shard_jobs)

shard_results = sorted(shard_results, key=lambda item: item["global_shard_index"])

# Merge JSONL shards in deterministic order.
with jsonl_path.open("w", encoding="utf-8") as merged_jsonl:
    for result in shard_results:
        with Path(result["jsonl_path"]).open("r", encoding="utf-8") as shard_file:
            shutil.copyfileobj(shard_file, merged_jsonl)

# Optional CSV merge. Disabled by default because it roughly doubles I/O cost.
if WRITE_CSV:
    with csv_path.open("w", encoding="utf-8", newline="") as merged_csv:
        writer = csv.DictWriter(merged_csv, fieldnames=csv_fieldnames)
        writer.writeheader()
        for result in shard_results:
            with Path(result["csv_path"]).open("r", encoding="utf-8", newline="") as shard_csv:
                reader = csv.DictReader(shard_csv)
                for row in reader:
                    writer.writerow(row)

with shard_manifest_path.open("w", encoding="utf-8") as manifest_file:
    for result in shard_results:
        manifest_file.write(json.dumps(result, ensure_ascii=False) + "\n")

total_rows = sum(result["rows_written"] for result in shard_results)
review_samples = {}
stats = {
    "target_counts": REGIME_COUNTS,
    "accepted_by_module": collections.Counter(),
    "accepted_by_regime": collections.Counter(),
    "accepted_by_module_regime": collections.Counter(),
    "duplicates_skipped": 0,
    "length_dropped": 0,
    "generator_failures": 0,
    "generator_failure_examples": [],
    "cot_failures": 0,
    "cot_failure_examples": [],
}

for result in shard_results:
    module_name = result["module"]
    regime = result["regime"]
    rows_written = result["rows_written"]
    stats["accepted_by_module"][module_name] += rows_written
    stats["accepted_by_regime"][regime] += rows_written
    stats["accepted_by_module_regime"][(module_name, regime)] += rows_written
    stats["duplicates_skipped"] += result["duplicates_skipped"]
    stats["length_dropped"] += result["length_dropped"]
    stats["generator_failures"] += result["generator_failures"]
    stats["cot_failures"] += result["cot_failures"]
    if module_name not in review_samples and result["review_sample"] is not None:
        review_samples[module_name] = result["review_sample"]
    for failure in result["cot_failure_examples"]:
        if len(stats["cot_failure_examples"]) < 25:
            stats["cot_failure_examples"].append(failure)
    for failure in result["generator_failure_examples"]:
        if len(stats["generator_failure_examples"]) < 25:
            stats["generator_failure_examples"].append(failure)

total_rows, review_samples


In [ ]:
# Dataset-level validation.

expected_total = sum(REGIME_COUNTS.values()) * len(TARGET_MODULES)
assert total_rows == expected_total, (total_rows, expected_total)

for module_name in TARGET_MODULES:
    assert module_name in review_samples, module_name
    for regime, expected_count in REGIME_COUNTS.items():
        count = stats["accepted_by_module_regime"][(module_name, regime)]
        assert count == expected_count, (module_name, regime, count, expected_count)

assert sum(stats["accepted_by_module"].values()) == expected_total
assert set(stats["accepted_by_regime"]) == set(REGIME_COUNTS)

print("Validation passed.")
print("Rows:", total_rows)
print("Generator failures during rejection sampling:", stats["generator_failures"])
print("CoT failures during rejection sampling:", stats["cot_failures"])
print("Duplicates skipped:", stats["duplicates_skipped"])

In [ ]:
# Save summary JSON and rejection examples. JSONL/CSV are streamed during generation.

summary = {
    "source": SOURCE_NAME,
    "base_seed": BASE_SEED,
    "configured_seed": SEED,
    "num_workers": NUM_WORKERS,
    "rows_per_shard": ROWS_PER_SHARD,
    "shard_count": len(shard_results),
    "write_csv": WRITE_CSV,
    "dedupe_scope": "within_shard",
    "target_modules": TARGET_MODULES,
    "regime_counts": REGIME_COUNTS,
    "total_rows": total_rows,
    "module_counts": dict(stats["accepted_by_module"]),
    "regime_counts_actual": dict(stats["accepted_by_regime"]),
    "module_regime_counts": {
        f"{module}::{regime}": count
        for (module, regime), count in stats["accepted_by_module_regime"].items()
    },
    "duplicates_skipped": stats["duplicates_skipped"],
    "length_dropped": stats["length_dropped"],
    "generator_failures": stats["generator_failures"],
    "generator_failure_examples": stats["generator_failure_examples"],
    "cot_failures": stats["cot_failures"],
    "strict_synthetic_validation": STRICT_SYNTHETIC_VALIDATION,
    "outputs": {
        "jsonl": str(jsonl_path),
        "csv": str(csv_path) if WRITE_CSV else None,
        "summary": str(summary_path),
        "cot_failure_examples": str(failure_path),
        "shard_manifest": str(shard_manifest_path),
    },
}

summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
failure_path.write_text(json.dumps(stats["cot_failure_examples"], ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved:")
print(" -", jsonl_path.resolve())
if WRITE_CSV:
    print(" -", csv_path.resolve())
print(" -", summary_path.resolve())
print(" -", failure_path.resolve())
print(" -", shard_manifest_path.resolve())

In [ ]:
# Human review preview: one retained row per target module.

preview = pd.DataFrame(
    {
        "module": row["module"],
        "regime": row["regime"],
        "input": row["input"],
        "cot": row["output"]["cot"],
        "answer": row["output"]["answer"],
    }
    for row in review_samples.values()
).sort_values("module").reset_index(drop=True)

preview